# Monte Carlo Convergence Study

This notebook analyzes the convergence properties of Monte Carlo estimators for option pricing.

## Topics Covered:
1. Convergence Rate Analysis
2. Standard Error Behavior
3. Sample Size Requirements
4. Convergence with Variance Reduction
5. Confidence Interval Coverage

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
sys.path.insert(0, '../src')

from mc_pricing import (
    EuropeanOption, BlackScholes,
    MonteCarloSimulator, ConvergenceAnalyzer, Visualizer
)
from mc_pricing.variance_reduction import AntitheticVariates, ControlVariates

%matplotlib inline
plt.style.use('seaborn-v0_8')

## 1. Theoretical Background

Monte Carlo estimators converge at rate O(1/√n):

$$\text{Standard Error} = \frac{\sigma}{\sqrt{n}}$$

where:
- σ is the standard deviation of payoffs
- n is the number of simulations

To reduce error by factor of 10, need 100x more samples!

In [ ]:
# Define test option
option = EuropeanOption(
    option_type='call',
    strike=100.0,
    maturity=1.0,
    spot=100.0,
    rate=0.05,
    volatility=0.2
)

# Black-Scholes benchmark
bs_price = BlackScholes.price(
    'call', 100.0, 100.0, 1.0, 0.05, 0.2
)

print(f"Black-Scholes Price: ${bs_price:.4f}")
print(f"\nThis serves as our 'true' price for convergence analysis.")

## 2. Convergence Test - Standard Monte Carlo

In [ ]:
# Test different sample sizes
n_trials = np.array([500, 1000, 2000, 5000, 10000, 20000, 50000, 100000, 200000, 500000])

print("Running convergence analysis...")
print("This may take a minute...\n")

analyzer = ConvergenceAnalyzer()
mean_prices, std_prices, std_errors = analyzer.analyze_convergence(
    option, n_trials, n_runs=10, seed=42
)

# Display results
print(f"{'N Simulations':<15} {'Mean Price':<12} {'Std Dev':<12} {'Std Error':<12} {'Error vs BS':<12}")
print("-" * 70)
for n, price, std, se in zip(n_trials, mean_prices, std_prices, std_errors):
    error_bs = abs(price - bs_price)
    print(f"{n:<15,} ${price:<11.4f} ${std:<11.4f} ${se:<11.4f} ${error_bs:<11.4f}")

In [ ]:
# Plot convergence
viz = Visualizer()
viz.plot_convergence(
    n_trials,
    mean_prices,
    std_errors,
    true_price=bs_price,
    title="Monte Carlo Convergence - European Call Option"
)

## 3. Verify O(1/√n) Convergence Rate

In [ ]:
# Fit power law to standard errors
log_n = np.log(n_trials)
log_se = np.log(std_errors)

# Linear regression in log space: log(SE) = a + b*log(n)
coeffs = np.polyfit(log_n, log_se, deg=1)
exponent = coeffs[0]

print(f"Estimated convergence rate: O(n^{exponent:.4f})")
print(f"Theoretical rate: O(n^-0.5)")
print(f"Difference: {abs(exponent - (-0.5)):.4f}")

if abs(exponent - (-0.5)) < 0.05:
    print("\n✓ Convergence rate matches theory!")
else:
    print("\n⚠ Convergence rate differs from theory (may need more trials)")

# Plot log-log
plt.figure(figsize=(10, 6))
plt.loglog(n_trials, std_errors, 'o-', label='Observed', markersize=8, linewidth=2)

# Theoretical O(1/sqrt(n))
theoretical = std_errors[0] * np.sqrt(n_trials[0] / n_trials)
plt.loglog(n_trials, theoretical, '--', label='Theoretical O(1/√n)', linewidth=2)

# Fitted power law
fitted = np.exp(coeffs[1]) * n_trials**coeffs[0]
plt.loglog(n_trials, fitted, ':', label=f'Fitted O(n^{exponent:.3f})', linewidth=2)

plt.xlabel('Number of Simulations', fontsize=12)
plt.ylabel('Standard Error', fontsize=12)
plt.title('Convergence Rate Verification (Log-Log Scale)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Sample Size Requirements

How many samples needed for different accuracy levels?

In [ ]:
# Estimate required sample sizes
# Estimate std from a pilot run
sim = MonteCarloSimulator(n_simulations=10000, seed=42)
_, se_pilot = sim.price(option)
estimated_std = se_pilot * np.sqrt(10000)

print(f"Estimated standard deviation of payoffs: ${estimated_std:.4f}\n")

# Calculate required samples for different error targets
target_errors = [0.01, 0.05, 0.10, 0.20, 0.50]

print(f"{'Target Error':<15} {'Required Samples':<20} {'Approx Time (100k/sec)':<25}")
print("-" * 65)

for target in target_errors:
    # For 95% confidence: SE = error / 1.96
    target_se = target / 1.96
    n_required = int(np.ceil((estimated_std / target_se) ** 2))
    time_estimate = n_required / 100000  # seconds at 100k sims/sec
    
    print(f"±${target:<14.2f} {n_required:<20,} {time_estimate:<25.2f}s")

## 5. Convergence with Variance Reduction

In [ ]:
# Compare convergence: Standard vs Antithetic vs Control Variates
print("Testing convergence with variance reduction...\n")

n_trials_vr = np.array([1000, 5000, 10000, 50000, 100000])

# Standard MC
standard_prices = []
standard_errors = []

for n in n_trials_vr:
    sim = MonteCarloSimulator(n_simulations=n, seed=42)
    price, se = sim.price(option)
    standard_prices.append(price)
    standard_errors.append(se)

# Antithetic Variates
av_prices = []
av_errors = []

for n in n_trials_vr:
    sim = MonteCarloSimulator(n_simulations=n, seed=42)
    price, se = sim.price(option, variance_reduction=AntitheticVariates())
    av_prices.append(price)
    av_errors.append(se)

# Control Variates
cv_prices = []
cv_errors = []

for n in n_trials_vr:
    sim = MonteCarloSimulator(n_simulations=n, seed=42)
    price, se = sim.price(option, variance_reduction=ControlVariates())
    cv_prices.append(price)
    cv_errors.append(se)

# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Price convergence
ax1.plot(n_trials_vr, standard_prices, 'o-', label='Standard MC', markersize=8, linewidth=2)
ax1.plot(n_trials_vr, av_prices, 's-', label='Antithetic Variates', markersize=8, linewidth=2)
ax1.plot(n_trials_vr, cv_prices, '^-', label='Control Variates', markersize=8, linewidth=2)
ax1.axhline(y=bs_price, color='r', linestyle='--', label='Black-Scholes', linewidth=2)
ax1.set_xscale('log')
ax1.set_xlabel('Number of Simulations', fontsize=12)
ax1.set_ylabel('Option Price ($)', fontsize=12)
ax1.set_title('Price Convergence Comparison', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Error convergence
ax2.loglog(n_trials_vr, standard_errors, 'o-', label='Standard MC', markersize=8, linewidth=2)
ax2.loglog(n_trials_vr, av_errors, 's-', label='Antithetic Variates', markersize=8, linewidth=2)
ax2.loglog(n_trials_vr, cv_errors, '^-', label='Control Variates', markersize=8, linewidth=2)
ax2.set_xlabel('Number of Simulations', fontsize=12)
ax2.set_ylabel('Standard Error ($)', fontsize=12)
ax2.set_title('Error Convergence Comparison', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Effective sample size
print("\nEffective Sample Size Multiplier:")
print("(How many standard MC samples = N variance-reduced samples)\n")
print(f"{'N Samples':<15} {'Antithetic':<15} {'Control':<15}")
print("-" * 45)
for i, n in enumerate(n_trials_vr):
    av_mult = (standard_errors[i] / av_errors[i]) ** 2
    cv_mult = (standard_errors[i] / cv_errors[i]) ** 2
    print(f"{n:<15,} {av_mult:<15.2f}x {cv_mult:<15.2f}x")

## 6. Confidence Interval Coverage

Test if 95% confidence intervals actually contain the true price 95% of the time.

In [ ]:
# Run multiple independent trials
n_sims = 50000
n_independent_trials = 100

print(f"Running {n_independent_trials} independent trials with {n_sims:,} simulations each...\n")

coverage_count = 0
prices = []
lower_bounds = []
upper_bounds = []

for i in range(n_independent_trials):
    sim = MonteCarloSimulator(n_simulations=n_sims, seed=1000+i)
    price, lower, upper = sim.price_with_confidence_interval(option, confidence=0.95)
    
    prices.append(price)
    lower_bounds.append(lower)
    upper_bounds.append(upper)
    
    if lower <= bs_price <= upper:
        coverage_count += 1

empirical_coverage = coverage_count / n_independent_trials

print(f"Empirical Coverage: {empirical_coverage*100:.1f}%")
print(f"Theoretical Coverage: 95.0%")
print(f"Difference: {abs(empirical_coverage - 0.95)*100:.1f}%")

if abs(empirical_coverage - 0.95) < 0.05:
    print("\n✓ Coverage is within acceptable range!")
else:
    print("\n⚠ Coverage differs from theory")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Price distribution
ax1.hist(prices, bins=30, alpha=0.7, edgecolor='black')
ax1.axvline(bs_price, color='r', linestyle='--', linewidth=2, label='True Price')
ax1.axvline(np.mean(prices), color='g', linestyle='-', linewidth=2, label='Mean Estimate')
ax1.set_xlabel('Price Estimate ($)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Distribution of MC Price Estimates', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Confidence intervals
trial_indices = range(min(50, n_independent_trials))  # Show first 50
for i in trial_indices:
    color = 'green' if lower_bounds[i] <= bs_price <= upper_bounds[i] else 'red'
    ax2.plot([lower_bounds[i], upper_bounds[i]], [i, i], color=color, alpha=0.5, linewidth=1.5)
    ax2.plot(prices[i], i, 'o', color=color, markersize=4)

ax2.axvline(bs_price, color='blue', linestyle='--', linewidth=2, label='True Price')
ax2.set_xlabel('Price ($)', fontsize=12)
ax2.set_ylabel('Trial Number', fontsize=12)
ax2.set_title(f'95% Confidence Intervals (First 50 trials)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Practical Guidelines

In [ ]:
# Create practical guidelines table
guidelines = pd.DataFrame({
    'Use Case': [
        'Quick estimate',
        'Standard pricing',
        'Production pricing',
        'High accuracy',
        'Research/validation'
    ],
    'Simulations': [
        '10,000',
        '50,000 - 100,000',
        '100,000 - 500,000',
        '500,000 - 1,000,000',
        '1,000,000+'
    ],
    'Expected Error (95% CI)': [
        '±$0.20 - $0.30',
        '±$0.10 - $0.15',
        '±$0.05 - $0.10',
        '±$0.02 - $0.05',
        '±$0.01 - $0.02'
    ],
    'Variance Reduction': [
        'Optional',
        'Recommended',
        'Strongly Recommended',
        'Required',
        'Required'
    ]
})

print("\nPractical Guidelines for Sample Size Selection:")
print("=" * 90)
print(guidelines.to_string(index=False))
print("=" * 90)

print("\nKey Recommendations:")
print("1. Always use variance reduction for production pricing")
print("2. 100,000 simulations is a good default for most applications")
print("3. Increase samples by 100x to reduce error by 10x")
print("4. Monitor convergence - don't blindly trust results")
print("5. Use antithetic variates as baseline (minimal overhead)")

## Summary

### Key Findings:

1. **Convergence Rate**: Confirmed O(1/√n) behavior
2. **Sample Size**: 100k simulations gives ~$0.05 error for typical ATM options
3. **Variance Reduction**: Can reduce effective samples needed by 2-10x
4. **Confidence Intervals**: 95% CIs have correct coverage
5. **Practical Guideline**: Use 100k-500k simulations with variance reduction

### Trade-offs:
- More samples = Better accuracy but slower
- Variance reduction = Better efficiency but more complex
- Optimal choice depends on accuracy requirements

Next: Check out `05_smile_calibration.ipynb` for volatility smile fitting!